# Day 15 — My First RAG Pipeline
Connect the Day 14 FAISS semantic search to an LLM and measure whether retrieval reduces hallucination.

**Stack:** `all-MiniLM-L6-v2` (embeddings, same as Days 13-14) + FAISS + LangChain `PromptTemplate` + a free local chat model (`Qwen2.5-1.5B-Instruct`). No API key needed. An OpenAI swap-in is shown in section 3.

**Runtime:** Colab → Runtime → Change runtime type → **T4 GPU** (the LLM is much faster on GPU).

In [1]:
!pip install -q faiss-cpu sentence-transformers langchain-core transformers accelerate numpy pandas

## 1. Knowledge base: a fictional company the LLM has never seen
The task wants facts the model has *no training knowledge of*, so I invented a company, **Veridia Labs**. Each line is one chunk.

In [2]:
knowledge_base = [
    'Veridia Labs was founded in 2019 in Srinagar by Dr. Hana Mir and Taj Bhat.',
    'The headquarters of Veridia Labs is on the third floor of the Dal View Building in Srinagar.',
    "Veridia's flagship product is Kesar, a crop-yield forecasting app built for saffron farmers.",
    'Kesar costs 499 rupees per month for the Basic plan and 1,299 rupees per month for the Pro plan.',
    'The Pro plan refreshes satellite imagery every 3 days, while the Basic plan refreshes it every 14 days.',
    'Refund policy: the Basic plan can be refunded within 7 days of purchase and the Pro plan within 14 days.',
    'Customer support is available Monday to Friday, 9 am to 6 pm IST, with no weekend support.',
    'As of 2026, Veridia Labs employs 42 people, of whom 12 are engineers.',
    'The CEO of Veridia Labs is Dr. Hana Mir and the CTO is Taj Bhat.',
    "Kesar's forecasting model, Zafran-2, was trained on 6 years of harvest records from 1,200 local farms.",
    'Zafran-2 predicts harvest yield with an error of about 8 percent on validation data.',
    'The Pro plan includes API access limited to 5,000 requests per day.',
    'The Basic plan does not include any API access.',
    'Farm data is stored on servers in the Mumbai region and is deleted within 30 days after an account is closed.',
    'Veridia Labs reported annual revenue of 3.2 crore rupees in 2025.',
    'Veridia Labs opened a second office in Anantnag in March 2026.',
    'Farmers who refer a friend to Kesar receive one free month of the Basic plan.',
    'Kesar supports three languages: Urdu, Kashmiri and English.',
    'Scheduled maintenance for Kesar happens on Sundays from 2 am to 4 am IST.',
    'The annual plan costs the same as 10 months of the monthly price, so customers get 2 months free.',
]
print(len(knowledge_base), 'chunks')

20 chunks


## 2. Retrieval layer (Day 14, reused): embed → FAISS index → `retrieve()`

In [3]:
import numpy as np, faiss, textwrap, pandas as pd
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

def embed(texts):
    return np.asarray(embedder.encode(list(texts), normalize_embeddings=True), dtype="float32")

kb_vectors = embed(knowledge_base)
index = faiss.IndexFlatL2(kb_vectors.shape[1])
index.add(kb_vectors)
print("vectors:", kb_vectors.shape, kb_vectors.dtype, "| index size:", index.ntotal)

def retrieve(query, top_k=3):
    D, I = index.search(embed([query]), top_k)
    return [(int(i), float(d), knowledge_base[int(i)]) for d, i in zip(D[0], I[0])]   # (chunk_id, L2 distance, text)

for r in retrieve("How much is the Basic plan?"): print(r)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

vectors: (20, 384) float32 | index size: 20
(3, 1.1008789539337158, 'Kesar costs 499 rupees per month for the Basic plan and 1,299 rupees per month for the Pro plan.')
(12, 1.1158723831176758, 'The Basic plan does not include any API access.')
(16, 1.1162036657333374, 'Farmers who refer a friend to Kesar receive one free month of the Basic plan.')


## 3. The LLM
Free local model, so no account or key. The only thing the rest of the notebook needs is a function `call_llm(system, user)`.

*Optional OpenAI swap:* replace the body of `call_llm` with `OpenAI().chat.completions.create(model=..., messages=[{"role":"system","content":system},{"role":"user","content":user}], temperature=0).choices[0].message.content` and nothing else changes.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
tok = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype="auto", device_map="auto")

def call_llm(system, user, max_new_tokens=150):
    messages = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)   # greedy = repeatable
    return tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(call_llm("You are a helpful assistant.", "Say hello in five words."))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Hello there! How can I assist you today?


## 4. The two functions the task asks for
**Structured prompt:** a LangChain `PromptTemplate` that keeps the CONTEXT and the QUESTION in clearly separated sections, plus a system rule: answer only from the context, otherwise say you don't know.

In [5]:
from langchain_core.prompts import PromptTemplate

RAG_SYSTEM = ("You are a careful assistant. Answer the question using ONLY the information in the CONTEXT. "
              "If the CONTEXT does not contain the answer, reply exactly: I don't know based on the provided documents.")
rag_prompt = PromptTemplate.from_template(
    "### CONTEXT\n{context}\n\n### QUESTION\n{question}\n\n### ANSWER (use only the context above)"
)

def format_context(chunks):
    return "\n".join(f"[{n+1}] {text}" for n, (_, _, text) in enumerate(chunks))

def generate_with_rag(query, top_k=3, return_trace=False):
    chunks = retrieve(query, top_k)                                   # 1. retrieve top-3 chunks
    prompt = rag_prompt.format(context=format_context(chunks), question=query)   # 2. fill the template
    answer = call_llm(RAG_SYSTEM, prompt)                             # 3. generate a grounded answer
    return (answer, chunks, prompt) if return_trace else answer

def generate_without_rag(query):
    return call_llm("You are a helpful assistant. Answer the question concisely.", query)   # question only, no context

## 5. Five test queries (facts the LLM cannot know)
`expected` = the chunk id that holds the answer, so retrieval can be checked automatically.

In [6]:
test_queries = [
    ("How much does the Basic plan of Kesar cost per month?", [3]),
    ("How accurate is the Zafran-2 forecasting model?", [10]),
    ("How many API requests per day does the Pro plan allow?", [11]),
    ("When is customer support available?", [6]),
    ("Where is farm data stored and how long until it is deleted after an account is closed?", [13]),
]

def show(title, text): print(f"  {title}:"); print(textwrap.indent(textwrap.fill(text, 95), "    "))

rows = []
for q, expected in test_queries:
    ans_rag, chunks, _ = generate_with_rag(q, return_trace=True)
    ans_no = generate_without_rag(q)
    hit = any(c[0] in expected for c in chunks)
    print("=" * 100); print("QUERY:", q); print("  retrieved chunk ids:", [c[0] for c in chunks], "| answer chunk retrieved:", hit)
    show("WITHOUT RAG", ans_no); show("WITH RAG", ans_rag)
    rows.append({"query": q, "retrieval_hit": hit, "without_rag": ans_no, "with_rag": ans_rag})

comparison = pd.DataFrame(rows)
comparison

QUERY: How much does the Basic plan of Kesar cost per month?
  retrieved chunk ids: [3, 16, 19] | answer chunk retrieved: True
  WITHOUT RAG:
    The Basic plan of Kesar costs 10 INR per month.
  WITH RAG:
    The Basic plan of Kesar costs 499 rupees per month.
QUERY: How accurate is the Zafran-2 forecasting model?
  retrieved chunk ids: [10, 9, 2] | answer chunk retrieved: True
  WITHOUT RAG:
    The accuracy of the Zafran-2 forecasting model can vary depending on its specific application
    and the data it uses. Generally, such models aim to predict weather patterns with some level of
    reliability but may not be 100% accurate due to various factors including atmospheric
    conditions, sensor limitations, and computational errors. For precise predictions, more
    advanced or specialized models might be required.
  WITH RAG:
    The Zafran-2 forecasting model has an error of about 8 percent on validation data.
QUERY: How many API requests per day does the Pro plan allow?
  retrie

,query,retrieval_hit,without_rag,with_rag
0,How much does the Basic plan of Kesar cost per...,True,The Basic plan of Kesar costs 10 INR per month.,The Basic plan of Kesar costs 499 rupees per m...
1,How accurate is the Zafran-2 forecasting model?,True,The accuracy of the Zafran-2 forecasting model...,The Zafran-2 forecasting model has an error of...
2,How many API requests per day does the Pro pla...,True,"The Pro plan allows up to 10,000 API requests ...","The Pro plan allows 5,000 API requests per day."
3,When is customer support available?,True,"Customer support hours vary by company, but ty...",Customer support is available Monday to Friday...
4,Where is farm data stored and how long until i...,True,Farm data is typically stored on servers manag...,Farm data is stored on servers in the Mumbai r...


## 6. Hunting for RAG failures
These queries are designed to stress the pipeline: info that is **not** in the knowledge base, arithmetic over a chunk, and a question spanning several chunks. For each one, `trace()` prints what was retrieved so I can decide **why** it failed.

**How to classify a failure**
- The chunk that holds the answer is *not* in the top 3 → **retrieval failure** (wrong chunk).
- The right chunk *is* in the top 3 but the answer contradicts or goes beyond it → **generation drift**.

In [7]:
hunt_queries = [
    ("What is the refund window for the annual plan?", []),              # not in KB (only Basic/Pro refunds exist)
    ("How many staff work in the Anantnag office?", []),                  # not in KB (office exists, headcount does not)
    ("How many of Veridia's employees are not engineers?", [7]),          # needs arithmetic: 42 - 12
    ("Who runs the company, who founded it and where is it based?", [0, 1, 8]),   # spans three chunks
]

def trace(query, expected):
    ans, chunks, prompt = generate_with_rag(query, return_trace=True)
    print("=" * 100); print("QUERY:", query)
    print("expected chunk ids:", expected or "none (answer is not in the knowledge base)")
    for cid, dist, text in chunks: print(f"  retrieved [{cid:2d}] L2={dist:.3f}  {text}")
    show("RAG ANSWER", ans)
    print("  --> answer chunk in top 3:", any(c[0] in expected for c in chunks) if expected else "n/a")

for q, e in hunt_queries: trace(q, e)

QUERY: What is the refund window for the annual plan?
expected chunk ids: none (answer is not in the knowledge base)
  retrieved [ 5] L2=0.866  Refund policy: the Basic plan can be refunded within 7 days of purchase and the Pro plan within 14 days.
  retrieved [19] L2=1.059  The annual plan costs the same as 10 months of the monthly price, so customers get 2 months free.
  retrieved [12] L2=1.333  The Basic plan does not include any API access.
  RAG ANSWER:
    The annual plan has a refund window of 14 days from the date of purchase.
  --> answer chunk in top 3: n/a
QUERY: How many staff work in the Anantnag office?
expected chunk ids: none (answer is not in the knowledge base)
  retrieved [15] L2=1.041  Veridia Labs opened a second office in Anantnag in March 2026.
  retrieved [ 7] L2=1.189  As of 2026, Veridia Labs employs 42 people, of whom 12 are engineers.
  retrieved [ 8] L2=1.363  The CEO of Veridia Labs is Dr. Hana Mir and the CTO is Taj Bhat.
  RAG ANSWER:
    Based on the in

## 7. Failure analysis

Across the 5 main queries, RAG answered all 5 from the knowledge base, while the no-RAG model invented a price (10 INR), an API limit (10,000 requests/day) and a storage provider (Microsoft). The two failures below came from the hunt queries.

**Failure 1: "What is the refund window for the annual plan?"** (answer is not in the knowledge base)
- Retrieved: chunks 5, 19, 12. Chunk 5 is the refund policy for the Basic (7 days) and Pro (14 days) plans.
- RAG answer: "14 days from the date of purchase." (wrong: the knowledge base never states an annual-plan refund window)
- Cause: **generation drift.** Retrieval did its job, because the closest chunk was the refund policy (L2 = 0.866). But that chunk does not cover the annual plan, and the model borrowed the Pro plan's 14 days instead of replying "I don't know", which my prompt explicitly told it to do.

**Failure 2: "How many staff work in the Anantnag office?"** (office exists in the knowledge base, headcount does not)
- Retrieved: chunks 15, 7, 8. Chunk 15 (the Anantnag office) is correct, but the closest chunk was already weak (L2 = 1.041).
- RAG answer: "12 staff members ... inferred from statement [2]". (wrong: 12 is the company-wide number of engineers, not the Anantnag headcount)
- Cause: **generation drift.** The right chunk was retrieved, but the model mixed two unrelated facts and cited a supporting statement that does not support its answer.

**Bonus (false refusal):** "How many of Veridia's employees are not engineers?" retrieved the right chunk first (L2 = 0.380: 42 employees, 12 engineers) but the model said "I don't know" instead of computing 42 - 12 = 30. Retrieval was perfect, but a 1.5B model failed at simple reasoning over the context.

**What this shows:** in all three cases the retriever found the right neighbourhood. The weak link was the generator, a small 1.5B-parameter model. Retrieval reduces hallucination a lot, but it doesn't guarantee a correct answer.

**Possible fixes**
- A similarity threshold, so the model gets no context when the best chunk is too far away (this would have helped Failure 2, but not Failure 1, where the closest chunk was fairly close).
- Ask the model to quote the exact sentence it used and reject answers that don't match the context.
- A larger model (or an OpenAI model) that follows the "only use the context" rule more reliably.
- Split chunks so one chunk holds one answerable fact, and retrieve more chunks for multi-part questions.
